In [1]:
import json
import os

from dotenv import load_dotenv
from pinecone import Pinecone
from openai import OpenAI

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

print("Pinecone client successfully configured.")
print(pinecone_api_key[:5] + "...")

openai_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=openai_api_key)

print("OpenAI client successfully configured.")
print(openai_api_key[:5] + "...")

Pinecone client successfully configured.
pcsk_...
OpenAI client successfully configured.
sk-pr...


In [2]:
def classify_intent(query,openai_client):
    """
    Classifies the intent of a query into 'factual', 'explanation', or 'guidance'.
    """
    messages = [{"role": "system", "content": f"Classify this query into 'factual', 'explanation', or 'guidance': {query}"}]
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=5
    )
    return response.choices[0].message.content.strip().lower()

In [3]:
def generate_prompt(query, intent):
    if intent == 'factual':
        return f"Provide a factual answer to the question: {query}"
    elif intent == 'explanation':
        return f"Explain in detail: {query}"
    elif intent == 'guidance':
        return f"Provide guidance for the following situation: {query}"
    else:
        return f"Answer the question: {query}"

In [4]:
def prompt_routing_rag(query,openai_client):
    intent = classify_intent(query,openai_client)
    print(f"Detected Intent: {intent}")

    prompt = generate_prompt(query, intent)
    print(f"Generated Prompt: {prompt}")

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": prompt}],
        max_tokens=250
    )
    return response.choices[0].message.content

In [5]:
query = "What is quantum entanglement?"
response = prompt_routing_rag(query,client)
print("RAG Response:", response)

Detected Intent: factual
Generated Prompt: Provide a factual answer to the question: What is quantum entanglement?
RAG Response: Quantum entanglement is a physical phenomenon that occurs when pairs or groups of particles become interconnected in such a way that the quantum state of one particle cannot be described independently of the state of the other(s), even when the particles are separated by large distances. This means that measuring the state of one particle will instantaneously determine the state of the other entangled particle, regardless of the distance between them. Entanglement is a key feature of quantum mechanics and has been experimentally verified numerous times. It plays a crucial role in quantum computing, quantum cryptography, and quantum teleportation.
